# Demo: reading hive-partitioned vs non-partitioned parquet with Polars


In [ ]:
import polars as pl
from pathlib import Path

DATA_DIR = Path("../data")
SINGLE_FILE = DATA_DIR / "sales.parquet"
PARTITIONED_DIR = DATA_DIR / "sales_partitioned"

## `read_parquet` vs `scan_parquet`

- **Predicate Pushdown**: Filters are pushed down to the file reader, so only matching rows are loaded
- **Projection Pushdown**: Only requested columns are loaded from disk, not the entire row


### Eager Reading with `read_parquet`


In [ ]:
# When you use `read_parquet`, Polars immediately loads the entire file into memory.
# This is simple and direct, but can be inefficient for large files.
df_eager = pl.read_parquet(SINGLE_FILE)

print(f"Type: {type(df_eager)}")
print(f"Shape: {df_eager.shape}")
print(f"Size: {df_eager.estimated_size() / 1024 / 1024:.2f} MB")

### Lazy Reading with `scan_parquet`


In [ ]:
# When you use `scan_parquet`, Polars creates a query plan but doesn't execute it yet.
# The actual data is only loaded when you call `.collect()`

# This returns instantly regardless of file size
lf = pl.scan_parquet(SINGLE_FILE)

print(f"Type: {type(lf)}")
# It does have metadata, like schema
print(lf.collect_schema())
# Query plan
print(lf.explain())
# To materialize the data in memory, call .collect()
df_from_lazy = lf.collect()
print(f"Shape: {df_from_lazy.shape}")

### Lazy Query Optimization


In [ ]:
# The real benefit of `scan_parquet` is that Polars can optimize your query before executing it.

# Example: We only want electronics sales from 2023, and only need a few columns
# With scan_parquet, Polars optimizes this query automatically

optimized_query = (
    pl.scan_parquet(SINGLE_FILE)
    .filter(pl.col("category") == "Electronics")
    .filter(pl.col("year") == 2023)
    .select(["product_name", "quantity", "total_amount", "region"])
)

print(optimized_query.explain())

**Notice the optimizations:**

- **Projection Pushdown**: Only the columns we need are listed (`product_name`, `quantity`, etc.)
- **Predicate Pushdown**: The filters (`category == "Electronics"`, `year == 2023`) are pushed down to the scan (Apply filters during reading, not after loading everything)


## Non-Partitioned vs Hive-Partitioned Data

### What is Hive Partitioning?

Hive partitioning organizes data into a directory structure where:

- Each level of directories represents a partition column
- The column name and value are encoded in the directory name

```
sales_partitioned/
├── year=2022/
│   ├── month=1/
│   │   └── *.parquet
│   ├── month=2/
│   │   └── *.parquet
│   └── ...
├── year=2023/
│   └── ...
└── year=2024/
    └── ...
```

### Benefits of Hive Partitioning

- **Partition Pruning**: When filtering by partition columns, entire directories can be skipped
- **Parallelism**: Multiple files can be read in parallel
- **Incremental Updates**: Add new partitions without rewriting existing data


In [ ]:
lf_partitioned = pl.scan_parquet(
    PARTITIONED_DIR / "**/*.parquet",
    hive_partitioning=True,
)

# Notice year and month are included from partition paths
print(lf_partitioned.collect_schema())

In [ ]:
# Partition pruning
january_2023_query = (
    pl.scan_parquet(
        PARTITIONED_DIR / "**/*.parquet",
        hive_partitioning=True,
    )
    .filter(pl.col("year") == 2023)
    .filter(pl.col("month") == 1)
)

print(january_2023_query.explain())